### 원하는 비율 조합 파일 생성 
- total reads: 5000 reads
- GBM percent: 0.1%, 0.5%, 1%, 2%, 2.5%, 3%, 5%, 10%, 100%

In [ ]:
import os
import random
import pysam
import subprocess

# 설정값
target_depth = 2500
mut_ratios = [0, 0.001, 0.005, 0.01, 0.02, 0.025, 0.03, 0.05, 0.1, 1]
num_files = 1000
BLOCK_SIZE = 100

mut_bam_path = "/Methylation/sample_data/step03_Preprocessing for ML/mut_sorted.bam"
wt_bam_path = "/Methylation/sample_data/step03_Preprocessing for ML/wt_sorted.bam"
base_output_dir = "/Methylation/results/step03_Preprocessing for ML/"
mut_name_file = "/Methylation/results/step03_Preprocessing for ML/mut_names.txt"
wt_name_file = "/Methylation/results/step03_Preprocessing for ML/wt_names.txt"

# BAM 참조 정보 및 리드 길이 계산
with pysam.AlignmentFile(mut_bam_path, "rb") as mut_bam:
    reference_lengths = dict(zip(mut_bam.references, mut_bam.lengths))
    sample_read = next(mut_bam.fetch(until_eof=True))
    read_length = sample_read.query_length

total_ref_len = sum(reference_lengths.values())
total_reads = total_ref_len * target_depth // read_length

# query name 저장 (최초 1회만 실행 필요)
def save_query_names(bam_path, output_txt):
    with pysam.AlignmentFile(bam_path, "rb") as bam, open(output_txt, "w") as f:
        names = set()
        for read in bam.fetch(until_eof=True):
            if read.is_paired and read.query_name not in names:
                f.write(read.query_name + "\n")
                names.add(read.query_name)

if not os.path.exists(mut_name_file):
    save_query_names(mut_bam_path, mut_name_file)

if not os.path.exists(wt_name_file):
    save_query_names(wt_bam_path, wt_name_file)

# 샘플링할 리드 수가 부족하면 중복을 허용하여 샘플링
def sample_query_names(name_file, k):
    with open(name_file, "r") as f:
        total_names = f.readlines()
    total_names = [name.strip() for name in total_names]
    # 샘플링할 수 있는 리드 수가 부족한 경우 중복을 허용하여 샘플링
    if k > len(total_names):
        print(f"Warning: 샘플링할 리드 수({k})가 가능한 리드 수({len(total_names)})보다 많습니다. 중복 샘플링이 이루어집니다.")
    return set(random.choices(total_names, k=k))  # 중복 샘플링 허용

# 블록 카운트 함수
def count_blocks(read, label, block_counts):
    if read.is_unmapped:
        return
    chrom = read.reference_name
    start = (read.reference_start // BLOCK_SIZE) * BLOCK_SIZE
    key = (chrom, start)
    if key not in block_counts:
        block_counts[key] = {"mut": 0, "wt": 0}
    block_counts[key][label] += 1

# 선택한 read 쌍 저장
def write_selected_pairs(bam_path, target_names_set, outbam, label, block_counts):
    written = set()
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam.fetch(until_eof=True):
            if read.query_name in target_names_set:
                if read.query_name not in written:
                    written.add(read.query_name)
                outbam.write(read)
                count_blocks(read, label, block_counts)

# 메인 루프
for mut_ratio in mut_ratios:
    folder_name = f"mut_{int(mut_ratio * 5000)}_reads"
    output_dir = os.path.join(base_output_dir, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    mut_pair_count = int((total_reads * mut_ratio) // 2)
    wt_pair_count = int(total_reads // 2) - mut_pair_count
    # 샘플링할 수 있는 리드 수가 가능한지 확인
    with open(mut_name_file, "r") as f:
        mut_names = f.readlines()
    with open(wt_name_file, "r") as f:
        wt_names = f.readlines()
    # 유효한 리드 수로 조정
    mut_pair_count = min(mut_pair_count, len(mut_names))
    wt_pair_count = min(wt_pair_count, len(wt_names))
    for i in range(num_files):
        sampled_mut_set = sample_query_names(mut_name_file, mut_pair_count)
        sampled_wt_set = sample_query_names(wt_name_file, wt_pair_count)
        output_bam_path = os.path.join(output_dir, f"sampled_reads_{i+1}.bam")
        header = {
            'HD': {'VN': '1.0', 'SO': 'coordinate'},
            'SQ': [{'SN': k, 'LN': v} for k, v in reference_lengths.items()]
        }
        block_counts = {}
        with pysam.AlignmentFile(output_bam_path, "wb", header=header) as outbam:
            write_selected_pairs(mut_bam_path, sampled_mut_set, outbam, "mut", block_counts)
            write_selected_pairs(wt_bam_path, sampled_wt_set, outbam, "wt", block_counts)
        stats_path = os.path.join(output_dir, f"block_read_counts_{i+1}.txt")
        with open(stats_path, "w") as f:
            f.write("chr\tstart\tend\tmutant_count\twt_count\n")
            for (chrom, start) in sorted(block_counts.keys()):
                end = start + BLOCK_SIZE
                mut_count = block_counts[(chrom, start)]["mut"]
                wt_count = block_counts[(chrom, start)]["wt"]
                f.write(f"{chrom}\t{start}\t{end}\t{mut_count}\t{wt_count}\n")
        print(f"[{mut_ratio:.3%}] 샘플 {i+1} 생성 완료: {output_bam_path}")

